# Global Solution 2026.2 · CitizenTech
## Triagem de solicitações de zeladoria urbana

**Disciplina** — AI for Robotic Process Automation

- Luiz Henrique Poss · RM562177
- Luis Filipe Chaves Amorim · RM564706
- Felipe Stefani Honorato · RM563380

O protótipo lê solicitações de um CSV, valida os campos, sugere uma categoria e uma prioridade, grava os resultados no SQLite e gera relatórios. A coluna `review_required` identifica as solicitações que precisam de avaliação humana.

## Problema e público

Moradores que encontram buracos, iluminação defeituosa, lixo acumulado ou barreiras de acessibilidade precisam informar o problema à zeladoria urbana. A equipe que recebe os relatos precisa ler descrições livres e identificar quais casos exigem atenção. O protótipo organiza essa triagem inicial e produz uma lista que a equipe pode filtrar por categoria, prioridade e necessidade de revisão.

## Temas da disciplina

| Tema | Onde entra no fluxo | Por que foi escolhido |
|---|---|---|
| Automação de processos (RPA) | `executar_pipeline` coordena leitura, validação, classificação, armazenamento e exportação. | Permite processar um lote inteiro sem repetir manualmente a triagem e a geração de relatórios. |
| Pandas e tratamento de dados | A leitura do CSV preserva os IDs como texto. A validação separa linhas incompletas. | Um registro com campo obrigatório vazio pode ser rejeitado sem descartar os demais. |
| PLN com regras | O texto é normalizado e comparado com termos de categoria e de risco. | A equipe consegue inspecionar por que a regra sinalizou um caso e revisar relatos com evidência insuficiente. |
| SQL e SQLite | A tabela usa `case_id` como chave e atualiza o registro quando o conteúdo muda. | A equipe pode consultar as solicitações armazenadas por categoria e prioridade sem criar cópias do mesmo caso ao reprocessar um lote. |

In [1]:
from contextlib import closing
from pathlib import Path
import logging
import re
import sqlite3
import tempfile
import unicodedata

import pandas as pd

## 1. Entrada de dados

Com `CSV_PROPRIO = None`, o notebook gera e usa uma amostra fictícia em `data/solicitacoes_demo.csv`. Para processar outro lote, informe o caminho do CSV na célula abaixo. Use dados sintéticos ou públicos sem dados pessoais.

In [2]:
CSV_PROPRIO = None

dados = [
    {'case_id': 'S001', 'description': 'Poste de iluminação apagado na Rua das Flores', 'neighborhood': 'Centro'},
    {'case_id': 'S002', 'description': 'Buraco grande na via principal', 'neighborhood': 'Jardim Azul'},
    {'case_id': 'S003', 'description': 'Lixo acumulado perto da praça', 'neighborhood': 'Vila Nova'},
    {'case_id': 'S004', 'description': 'Calçada sem rampa acessível', 'neighborhood': 'Boa Vista'},
    {'case_id': 'S005', 'description': 'Fio caído com risco de choque', 'neighborhood': 'Centro'},
    {'case_id': 'S006', 'description': 'Solicitação sobre situação não catalogada', 'neighborhood': 'Bairro Sul'},
]


PASTA_DADOS = Path('data')
PASTA_SAIDA = Path('outputs')
CSV_DEMO = PASTA_DADOS / 'solicitacoes_demo.csv'
CSV_ENTRADA = CSV_DEMO if CSV_PROPRIO is None else Path(CSV_PROPRIO)

## 2. Categoria, prioridade e revisão

A pontuação de uma categoria é o número de termos distintos encontrados no relato. Empates e descrições sem categoria ficam em `Outros`. Uma única correspondência indica evidência insuficiente para dispensar a revisão. Sinais explícitos de risco elevam a prioridade e também exigem revisão. A coluna `confidence` resume a força da correspondência com o vocabulário, em uma escala de 0 a 1.

In [3]:
COLUNAS_OBRIGATORIAS = {'case_id', 'description', 'neighborhood'}

PALAVRAS_POR_CATEGORIA = {
    'Iluminação pública': ['iluminacao', 'poste', 'postes', 'lampada', 'lampadas', 'luz apagada'],
    'Buraco ou pavimento': ['buraco', 'buracos', 'cratera', 'crateras', 'pavimento danificado'],
    'Resíduos e limpeza': ['lixo', 'entulho', 'residuos', 'coleta'],
    'Acessibilidade': ['rampa', 'rampas', 'calcada', 'calcadas', 'acessibilidade', 'piso tatil'],
}

CATEGORIAS_PRIORIDADE_MEDIA = ('Iluminação pública', 'Buraco ou pavimento', 'Resíduos e limpeza')
PADRAO_RISCO = r'\b(?:fio[s]? caido[s]?|fio[s]? exposto[s]?|fiacao exposta|risco de choque|faisca[s]?|alagamento[s]?|queda iminente)\b'
NEGACAO_EXPLICITA = r'\b(?:sem|nao ha) (?:risco de choque|faisca[s]?|alagamento[s]?)\b'

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize('NFKD', texto.lower())
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'[^\w\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def classificar_solicitacao(descricao: str) -> dict:
    texto = normalizar(descricao)
    pontuacoes = {
        categoria: sum(bool(re.search(rf'\b{re.escape(termo)}\b', texto)) for termo in termos)
        for categoria, termos in PALAVRAS_POR_CATEGORIA.items()
    }
    maior_pontuacao = max(pontuacoes.values())
    melhores = [categoria for categoria, pontos in pontuacoes.items() if pontos == maior_pontuacao]
    categoria = melhores[0] if maior_pontuacao and len(melhores) == 1 else 'Outros'
    confianca = min(maior_pontuacao / 2, 1.0) if categoria != 'Outros' else 0.0
    revisao = confianca < 1.0

    # shortcut: cobre negações literais; ampliar após avaliar relatos com outras formas de negação.
    texto_risco = re.sub(NEGACAO_EXPLICITA, '', texto)
    if re.search(PADRAO_RISCO, texto_risco):
        prioridade, revisao = 'Alta', True
    elif any(pontuacoes[categoria] for categoria in CATEGORIAS_PRIORIDADE_MEDIA):
        prioridade = 'Média'
    else:
        prioridade = 'Baixa'
    return {'category': categoria, 'priority': prioridade,
            'confidence': confianca, 'review_required': revisao}

## 3. Processamento do lote

O fluxo valida o cabeçalho e os campos obrigatórios. Para IDs repetidos no mesmo CSV, prevalece a primeira linha válida. Em lotes posteriores, uma alteração no mesmo ID atualiza o SQLite. O relatório CSV mostra as solicitações válidas do lote atual. O log registra a contagem de registros e as falhas.

In [4]:
COLUNAS_RELATORIO = ['case_id', 'description', 'neighborhood', 'category',
                    'priority', 'confidence', 'review_required']

def criar_banco(con: sqlite3.Connection) -> None:
    con.execute('''CREATE TABLE IF NOT EXISTS requests (
        case_id TEXT PRIMARY KEY, description TEXT NOT NULL, neighborhood TEXT NOT NULL,
        category TEXT NOT NULL, priority TEXT NOT NULL, confidence REAL NOT NULL,
        review_required INTEGER NOT NULL)''')

def executar_pipeline(caminho_csv: Path, pasta_saida: Path, caminho_banco: Path) -> dict:
    logger = logging.Logger('citizentech', logging.INFO)
    arquivo_log = None
    try:
        arquivos_gerados = (caminho_banco, pasta_saida / 'execucao.log',
                            pasta_saida / 'solicitacoes_processadas.csv',
                            pasta_saida / 'registros_rejeitados.csv')
        if caminho_csv.resolve() in {arquivo.resolve() for arquivo in arquivos_gerados}:
            mensagem = 'O CSV de entrada deve ter um caminho diferente dos arquivos gerados.'
            logging.getLogger('citizentech').error(mensagem)
            return {'status': 'erro', 'message': mensagem}
        pasta_saida.mkdir(parents=True, exist_ok=True)
        arquivo_log = logging.FileHandler(pasta_saida / 'execucao.log', encoding='utf-8')
        arquivo_log.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
        logger.addHandler(arquivo_log)

        df = pd.read_csv(caminho_csv, dtype=str, keep_default_na=False, encoding='utf-8-sig')
        faltantes = COLUNAS_OBRIGATORIAS - set(df.columns)
        if faltantes:
            mensagem = 'Colunas obrigatórias ausentes: ' + ', '.join(sorted(faltantes))
            logger.error(mensagem)
            return {'status': 'erro', 'message': mensagem}
        if not isinstance(df.index, pd.RangeIndex):
            mensagem = 'O CSV contém linhas com mais campos do que o cabeçalho.'
            logger.error(mensagem)
            return {'status': 'erro', 'message': mensagem}

        processados, rejeitados, ids_vistos = [], [], set()
        duplicados = 0
        for numero_linha, row in enumerate(df.to_dict(orient='records'), start=2):
            case_id = row['case_id'].strip()
            descricao = row['description'].strip()
            bairro = row['neighborhood'].strip()
            if not case_id or not descricao or not bairro:
                rejeitados.append({'row_number': numero_linha, 'case_id': case_id,
                                   'reason': 'case_id, description e neighborhood são obrigatórios'})
                continue
            if case_id in ids_vistos:
                duplicados += 1
                logger.info('Linha %s ignorada por ID repetido no lote', numero_linha)
                continue
            ids_vistos.add(case_id)
            processados.append({'case_id': case_id, 'description': descricao, 'neighborhood': bairro,
                                **classificar_solicitacao(descricao)})

        inseridos = atualizados = 0
        caminho_banco.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=pasta_saida) as pasta_temporaria:
            classificados = Path(pasta_temporaria) / 'solicitacoes_processadas.csv'
            invalidos = Path(pasta_temporaria) / 'registros_rejeitados.csv'
            pd.DataFrame(processados, columns=COLUNAS_RELATORIO).to_csv(
                classificados, index=False, encoding='utf-8')
            pd.DataFrame(rejeitados, columns=['row_number', 'case_id', 'reason']).to_csv(
                invalidos, index=False, encoding='utf-8')

            with closing(sqlite3.connect(caminho_banco)) as con, con:
                con.execute('BEGIN IMMEDIATE')
                criar_banco(con)
                for item in processados:
                    valores = tuple(item[coluna] for coluna in COLUNAS_RELATORIO)
                    anterior = con.execute('''SELECT description, neighborhood, category,
                        priority, confidence, review_required FROM requests WHERE case_id = ?''',
                        (item['case_id'],)).fetchone()
                    if anterior == valores[1:]:
                        continue
                    con.execute('''INSERT INTO requests
                        (case_id, description, neighborhood, category, priority, confidence, review_required)
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                        ON CONFLICT(case_id) DO UPDATE SET
                            description = excluded.description, neighborhood = excluded.neighborhood,
                            category = excluded.category, priority = excluded.priority,
                            confidence = excluded.confidence, review_required = excluded.review_required''',
                        valores)
                    inseridos += anterior is None
                    atualizados += anterior is not None
                invalidos.replace(pasta_saida / invalidos.name)
                classificados.replace(pasta_saida / classificados.name)

        logger.info('Lidos=%s válidos=%s novos=%s atualizados=%s rejeitados=%s duplicados=%s',
                    len(df), len(processados), inseridos, atualizados, len(rejeitados), duplicados)
        return {'status': 'ok', 'read': len(df), 'valid': len(processados), 'inserted': inseridos,
                'updated': atualizados, 'rejected': len(rejeitados), 'duplicates': duplicados,
                'review_required': sum(item['review_required'] for item in processados)}
    except (OSError, sqlite3.Error, pd.errors.EmptyDataError,
            pd.errors.ParserError, UnicodeDecodeError) as exc:
        if isinstance(exc, FileNotFoundError) and exc.filename and Path(exc.filename) == caminho_csv:
            mensagem = 'Arquivo de entrada não encontrado.'
        elif isinstance(exc, (pd.errors.EmptyDataError, pd.errors.ParserError, UnicodeDecodeError)):
            mensagem = f'CSV inválido ou ilegível: {exc}'
        else:
            mensagem = f'Falha no processamento: {exc}'
        destino_log = logger if arquivo_log is not None else logging.getLogger('citizentech')
        destino_log.error(mensagem)
        return {'status': 'erro', 'message': mensagem}
    finally:
        if arquivo_log is not None:
            arquivo_log.close()

## 4. Execução

Execute as células em ordem. A demonstração cria um CSV separado e usa o mesmo processamento disponível para um arquivo próprio. Quando o processamento retorna erro, a exibição dos relatórios e a consulta ao banco são puladas.

In [5]:
try:
    if CSV_PROPRIO is None:
        CSV_DEMO.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(dados).to_csv(CSV_DEMO, index=False, encoding='utf-8')
except OSError as exc:
    mensagem = f'Falha ao criar o CSV de demonstração: {exc}'
    logging.getLogger('citizentech').error(mensagem)
    resultado = {'status': 'erro', 'message': mensagem}
else:
    resultado = executar_pipeline(CSV_ENTRADA, PASTA_SAIDA, PASTA_DADOS / 'requests.sqlite')

resultado

{'status': 'ok',
 'read': 6,
 'valid': 6,
 'inserted': 6,
 'updated': 0,
 'rejected': 0,
 'duplicates': 0,
 'review_required': 4}

In [6]:
if resultado['status'] == 'ok':
    print('Solicitações classificadas')
    display(pd.read_csv(PASTA_SAIDA / 'solicitacoes_processadas.csv', dtype={'case_id': str}, keep_default_na=False))
    print('Registros rejeitados')
    display(pd.read_csv(PASTA_SAIDA / 'registros_rejeitados.csv', dtype={'case_id': str}, keep_default_na=False))
else:
    print(resultado['message'])

Solicitações classificadas


,case_id,description,neighborhood,category,priority,confidence,review_required
0,S001,Poste de iluminação apagado na Rua das Flores,Centro,Iluminação pública,Média,1.0,False
1,S002,Buraco grande na via principal,Jardim Azul,Buraco ou pavimento,Média,0.5,True
2,S003,Lixo acumulado perto da praça,Vila Nova,Resíduos e limpeza,Média,0.5,True
3,S004,Calçada sem rampa acessível,Boa Vista,Acessibilidade,Baixa,1.0,False
4,S005,Fio caído com risco de choque,Centro,Outros,Alta,0.0,True
5,S006,Solicitação sobre situação não catalogada,Bairro Sul,Outros,Baixa,0.0,True


Registros rejeitados


,row_number,case_id,reason


## 5. Validação

A amostra rotulada é sintética. Ela inclui relatos com mais de uma categoria possível, risco explícito, uma negação e descrições fora do vocabulário. A comparação mede categoria, prioridade e revisão humana separadamente. Os resultados descrevem apenas estes exemplos.

In [7]:
testes_rotulados = [
    ('A lâmpada do poste pisca desde ontem', 'Iluminação pública', 'Média', False),
    ('Uma cratera abriu no pavimento danificado', 'Buraco ou pavimento', 'Média', False),
    ('A coleta de resíduos parou na praça', 'Resíduos e limpeza', 'Média', False),
    ('Falta piso tátil na rampa do terminal', 'Acessibilidade', 'Baixa', False),
    ('Poste soltando faíscas ao lado da escola', 'Iluminação pública', 'Alta', True),
    ('Buraco perto de um poste', 'Outros', 'Média', True),
    ('Não há risco de choque, apenas lixo perto do terminal', 'Resíduos e limpeza', 'Média', True),
    ('Entulho na esquina', 'Resíduos e limpeza', 'Média', True),
    ('Quero consultar o horário da feira', 'Outros', 'Baixa', True),
    ('As luminárias do largo deixaram de acender', 'Iluminação pública', 'Média', True),
    ('A sarjeta virou um rio depois da chuva', 'Outros', 'Alta', True),
    ('A cadeira de rodas não passa pela entrada', 'Acessibilidade', 'Baixa', True),
]

avaliacao = []
for descricao, categoria, prioridade, revisao in testes_rotulados:
    previsto = classificar_solicitacao(descricao)
    avaliacao.append({
        'relato': descricao,
        'categoria esperada': categoria, 'categoria prevista': previsto['category'],
        'prioridade esperada': prioridade, 'prioridade prevista': previsto['priority'],
        'revisão esperada': revisao, 'revisão prevista': previsto['review_required'],
    })

comparacao = pd.DataFrame(avaliacao)
for campo in ('categoria', 'prioridade', 'revisão'):
    acertos = comparacao[f'{campo} esperada'].eq(comparacao[f'{campo} prevista']).sum()
    total = len(comparacao)
    print(f'{campo.capitalize()} · {acertos}/{total} acertos, {total - acertos} erros ({acertos/total:.0%})')
display(comparacao)

assert classificar_solicitacao(testes_rotulados[4][0])['priority'] == 'Alta'
assert classificar_solicitacao(testes_rotulados[4][0])['review_required']
empate = classificar_solicitacao(testes_rotulados[5][0])
assert empate['category'] == 'Outros' and empate['review_required']

Categoria · 10/12 acertos, 2 erros (83%)
Prioridade · 10/12 acertos, 2 erros (83%)
Revisão · 12/12 acertos, 0 erros (100%)


,relato,categoria esperada,categoria prevista,prioridade esperada,prioridade prevista,revisão esperada,revisão prevista
0,A lâmpada do poste pisca desde ontem,Iluminação pública,Iluminação pública,Média,Média,False,False
1,Uma cratera abriu no pavimento danificado,Buraco ou pavimento,Buraco ou pavimento,Média,Média,False,False
2,A coleta de resíduos parou na praça,Resíduos e limpeza,Resíduos e limpeza,Média,Média,False,False
3,Falta piso tátil na rampa do terminal,Acessibilidade,Acessibilidade,Baixa,Baixa,False,False
4,Poste soltando faíscas ao lado da escola,Iluminação pública,Iluminação pública,Alta,Alta,True,True
5,Buraco perto de um poste,Outros,Outros,Média,Média,True,True
6,"Não há risco de choque, apenas lixo perto do t...",Resíduos e limpeza,Resíduos e limpeza,Média,Média,True,True
7,Entulho na esquina,Resíduos e limpeza,Resíduos e limpeza,Média,Média,True,True
8,Quero consultar o horário da feira,Outros,Outros,Baixa,Baixa,True,True
9,As luminárias do largo deixaram de acender,Iluminação pública,Outros,Média,Baixa,True,True


In [8]:
with tempfile.TemporaryDirectory() as pasta_teste:
    pasta = Path(pasta_teste)
    banco_teste = pasta / 'teste.sqlite'
    saida_teste = pasta / 'relatorios'

    falha = executar_pipeline(pasta / 'ausente.csv', saida_teste, banco_teste)
    print('Arquivo ausente', falha)
    assert falha['status'] == 'erro'
    assert 'Arquivo de entrada não encontrado' in (saida_teste / 'execucao.log').read_text(encoding='utf-8')
    assert not (saida_teste / 'solicitacoes_processadas.csv').exists()

    entrada_teste = pasta / 'lote.csv'
    entrada_teste.write_text('case_id,description,neighborhood\n001,Poste apagado,Centro\n'
                            '002,,Centro\n001,Entulho,Centro\n', encoding='utf-8')
    lote = executar_pipeline(entrada_teste, saida_teste, banco_teste)
    assert (lote['inserted'], lote['rejected'], lote['duplicates']) == (1, 1, 1)
    repeticao = executar_pipeline(entrada_teste, saida_teste, banco_teste)
    assert repeticao['inserted'] == repeticao['updated'] == 0
    print('Lote misto', lote)
    print('Repetição', repeticao)

    entrada_teste.write_text('case_id,description,neighborhood\n001,Lixo na praça,Centro\n',
                            encoding='utf-8')
    alteracao = executar_pipeline(entrada_teste, saida_teste, banco_teste)
    assert alteracao['inserted'] == 0 and alteracao['updated'] == 1
    with closing(sqlite3.connect(banco_teste)) as con:
        registro = con.execute('SELECT case_id, description FROM requests').fetchone()
    exportado = pd.read_csv(saida_teste / 'solicitacoes_processadas.csv', dtype={'case_id': str})
    assert registro == tuple(exportado.loc[0, ['case_id', 'description']]) == ('001', 'Lixo na praça')
    print('ID atualizado no banco e no CSV', registro[0])

Arquivo ausente {'status': 'erro', 'message': 'Arquivo de entrada não encontrado.'}
Lote misto {'status': 'ok', 'read': 3, 'valid': 1, 'inserted': 1, 'updated': 0, 'rejected': 1, 'duplicates': 1, 'review_required': 1}
Repetição {'status': 'ok', 'read': 3, 'valid': 1, 'inserted': 0, 'updated': 0, 'rejected': 1, 'duplicates': 1, 'review_required': 1}
ID atualizado no banco e no CSV 001


## 6. Consulta ao banco

A consulta agrega todas as solicitações armazenadas no SQLite. O resumo abaixo se refere ao lote atual, enquanto o banco mantém os registros válidos de lotes anteriores.

In [9]:
if resultado['status'] == 'ok':
    with closing(sqlite3.connect(PASTA_DADOS / 'requests.sqlite')) as con:
        consulta = pd.read_sql_query(
            'SELECT category, priority, COUNT(*) AS quantidade FROM requests '
            'GROUP BY category, priority ORDER BY category, priority',
            con,
        )
    display(consulta)
    print('Resumo do lote', resultado)
    print('Arquivos gerados', sorted(p.name for p in PASTA_SAIDA.iterdir() if p.is_file()))

,category,priority,quantidade
0,Acessibilidade,Baixa,1
1,Buraco ou pavimento,Média,1
2,Iluminação pública,Média,1
3,Outros,Alta,1
4,Outros,Baixa,1
5,Resíduos e limpeza,Média,1


Resumo do lote {'status': 'ok', 'read': 6, 'valid': 6, 'inserted': 6, 'updated': 0, 'rejected': 0, 'duplicates': 0, 'review_required': 4}
Arquivos gerados ['execucao.log', 'registros_rejeitados.csv', 'solicitacoes_processadas.csv']


## Limites da triagem

Nos testes, “luminárias” e “cadeira de rodas” ficaram fora do vocabulário das respectivas categorias. O relato indireto de alagamento também ficou com prioridade baixa. Os três casos foram sinalizados para revisão. A lista de termos e o tratamento de negações são limitados. Uma equipe de zeladoria deve confirmar categoria e urgência antes de tomar decisões de atendimento.

## Proteção de dados, ética e ODS

A demonstração usa relatos fictícios e processa os arquivos localmente, sem enviar dados a serviços externos. O SQLite e o CSV de saída guardam a descrição original. Por isso, a entrada deve ficar livre de nomes, telefones, endereços residenciais e outros dados pessoais. Em um piloto real, seria preciso definir acesso, retenção e revisão humana. Organizar a triagem de zeladoria se relaciona com o ODS 11 (Cidades e Comunidades Sustentáveis). Dar visibilidade a barreiras de acessibilidade se relaciona com o ODS 10 (Redução das Desigualdades). O protótipo é acadêmico e não presta atendimento emergencial.